# Airbnb Price Prediction

Menggunakan model **LightGBM**, dengan fitur tambahan dari analisis sentimen
komentar tamu dan penggabungan beberapa tabel data (listing, host, review, comment).

**Alur kerja:**
1. Import library & load data
2. Feature engineering — sentimen komentar
3. Menggabungkan (merge) seluruh tabel data
4. Feature engineering — parsing kolom mentah menjadi fitur numerik
5. Menentukan kolom fitur (kategorikal & numerik)
6. Membersihkan target (`price`) & log-transform
7. Cross-validation (KFold) untuk evaluasi model
8. Training model final & membuat file submission


## 1. Import Library & Load Data

In [4]:
import os
import glob
import re
import ast

import numpy as np
import pandas as pd
from textblob import TextBlob
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_percentage_error


In [5]:
# Mencari adress file secara otomatis
CANDIDATE_DIRS = ["../data", "/kaggle/input"]
PATH = next((d for d in CANDIDATE_DIRS if os.path.isdir(d)), "../data")

train_path   = glob.glob(f'{PATH}/**/train*.csv', recursive=True)[0]
test_path    = glob.glob(f'{PATH}/**/test*.csv', recursive=True)[0]
airbnb_path  = glob.glob(f'{PATH}/**/airbnb*.csv', recursive=True)[0]
comment_path = glob.glob(f'{PATH}/**/comment*.csv', recursive=True)[0]
host_path    = glob.glob(f'{PATH}/**/host*.csv', recursive=True)[0]
review_path  = glob.glob(f'{PATH}/**/review*.csv', recursive=True)[0]

train_raw  = pd.read_csv(train_path)
test_raw   = pd.read_csv(test_path)
airbnb_df  = pd.read_csv(airbnb_path)
comment_df = pd.read_csv(comment_path)
host_df    = pd.read_csv(host_path)
review_df  = pd.read_csv(review_path)

# mengumpulkan dataframe ke dalam satu dictionary
datasets = {
    "train": train_raw,
    "test": test_raw,
    "host": host_df,
    "review": review_df,
    "comment": comment_df,
    "airbnb": airbnb_df,
}

# Looping
for name, df in datasets.items():
    print("=" * 50)
    print(name.upper())
    print("=" * 50)
    print("Shape:", df.shape)
    print("\nMissing value:")
    print(df.isnull().sum())
    print("\nHead:")
    display(df.head())


TRAIN
Shape: (2954, 2)

Missing value:
id       0
price    0
dtype: int64

Head:


,id,price
0,840338886577189778,88.0
1,25963531,76.0
2,15813099,202.0
3,1159877277633034540,531.0
4,630126688219312259,345.0


TEST
Shape: (739, 1)

Missing value:
id    0
dtype: int64

Head:


,id
0,39326791
1,1454846206172103024
2,711238794811729789
3,17519803
4,1334579376251132916


HOST
Shape: (3693, 12)

Missing value:
id                                 0
host_about                      1374
host_response_time               856
host_response_rate               856
host_acceptance_rate             915
host_neighbourhood               193
host_total_listings_count          0
host_verifications                 0
host_identity_verified             0
neighbourhood                   1973
neighbourhood_cleansed             0
neighbourhood_group_cleansed       0
dtype: int64

Head:


...

,id,number_of_reviews,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,reviews_per_month
0,46475943,4,5.0,5.0,4.75,5.0,4.75,5.0,4.5,0.10
1,864321103080953349,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,841098685339489415,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,52681365,2,4.5,5.0,4.50,4.5,5.00,5.0,4.5,0.09
4,638830141716356817,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


COMMENT
Shape: (38350, 5)

Missing value:
id                0
date              0
reviewer_id       0
reviewer_name     0
comments         12
dtype: int64

Head:


,id,date,reviewer_id,reviewer_name,comments
0,71609,2011-12-19,1456140,Max,The rooms were clean and tidy. Beds very comfo...
1,71609,2012-07-17,1804182,Jaiya,Good space and quite an interesting home in a ...
2,71609,2012-09-01,3113461,Zahra,It was a comfortable place. Belinda was a kind...
3,71609,2012-09-04,1432123,Helmut,We are four mature age travellers and stayed f...
4,71609,2013-01-02,2759938,Jack,Belinda is a great host! She helps you wheneve...


AIRBNB
Shape: (3693, 24)

Missing value:
id                           0
name                         0
description                 71
neighborhood_overview     1973
property_type                0
room_type                    0
accommodates                 0
bathrooms_text              10
bedrooms                   312
beds                       888
amenities                    0
minimum_nights               0
maximum_nights               0
instant_bookable             0
longitude                    0
latitude                     0
minimum_minimum_nights       1
maximum_minimum_nights       1
minimum_maximum_nights       1
maximum_maximum_nights       1
availability_30              0
availability_60              0
availability_90              0
availability_365             0
dtype: int64

Head:


,id,name,description,neighborhood_overview,property_type,room_type,accommodates,bathrooms_text,bedrooms,beds,...,longitude,latitude,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,availability_30,availability_60,availability_90,availability_365
0,71609,Ensuite Room (Room 1 & 2) near EXPO,For 3 rooms.Book room 1&2 and room 4,NaN,Private room in villa,Private room,2,1 private bath,2.0,NaN,...,103.95887,1.34537,92.0,92.0,1125.0,1125.0,30,60,90,90
1,71896,B&B Room 1 near Airport & EXPO,NaN,NaN,Private room in home,Private room,1,Shared half-bath,1.0,1.0,...,103.95958,1.34754,92.0,92.0,1125.0,1125.0,30,60,90,90
2,71903,Room 2-near Airport & EXPO,"Like your own home, 24hrs access.",Quiet and view of the playground with exercise...,Private room in home,Private room,2,Shared half-bath,1.0,2.0,...,103.96100,1.34531,92.0,92.0,1125.0,1125.0,30,60,90,90
3,275343,10min walk to MRT & a Cozy Room with window! (1),**IMPORTANT NOTES: READ BEFORE YOU BOOK! <br ...,NaN,Private room in rental unit,Private room,1,2 shared baths,NaN,NaN,...,103.80814,1.29015,180.0,180.0,1125.0,1125.0,0,0,0,0
4,275344,15 mins to Outram MRT Single Room (2),Lovely home for the special guest !,Bus stop <br />Food center <br />Supermarket,Private room in rental unit,Private room,1,2.5 shared baths,NaN,NaN,...,103.81144,1.28836,180.0,180.0,1125.0,1125.0,0,0,0,0


## 2. Feature Engineering — Sentimen Komentar

Menghitung skor sentimen tiap komentar dengan **TextBlob** (skor -1 = negatif,
1 = positif), lalu mengagregasinya per listing (`id`): rata-rata sentimen,
jumlah komentar, rata-rata panjang komentar, dan jumlah reviewer unik.


In [6]:
# isi missing value di comments dengan "" agar tidak mengubah nilai
comment_df["comments"] = comment_df["comments"].fillna("").astype(str)


def get_sentiment(text):
# mengembalikan skor sentimen berkisar antara -1 (negatif) sampai 1 (positif).
    return TextBlob(str(text)).sentiment.polarity

# menghitung sentiment tiap komentar
comment_df["sentiment"] = comment_df["comments"].apply(get_sentiment)

comment_feature = (
    comment_df.groupby("id")
    .agg(
        avg_sentiment=("sentiment", "mean"),
        comment_count=("comments", "size"),
        avg_comment_len=("comments", lambda x: x.str.len().mean()),
        n_unique_reviewers=("reviewer_id", "nunique"),
    )
    .reset_index()
)
comment_feature.head()


,id,avg_sentiment,comment_count,avg_comment_len,n_unique_reviewers
0,71609,0.357396,19,165.210526,19
1,71896,0.261404,24,351.541667,24
2,71903,0.314688,46,232.478261,46
3,275343,0.364191,20,250.300000,20
4,275344,0.287356,16,515.750000,16


## 3. Menggabungkan (Merge) Seluruh Tabel

Menggabungkan tabel `airbnb`, `host`, `review`, dan fitur komentar
(`comment_feature`) ke tabel `train`/`test` berdasarkan kolom `id`.


In [7]:
def build_features(df):
    df = df.merge(airbnb_df, on='id', how='left')
    df = df.merge(host_df, on='id', how='left')
    df = df.merge(review_df, on='id', how='left')
    df = df.merge(comment_feature, on='id', how='left')
    return df

train = build_features(train_raw.copy())
test  = build_features(test_raw.copy())


## 4. Feature Engineering — Parsing Kolom Mentah

Mengubah kolom teks/mentah menjadi fitur numerik yang bisa dipakai model.

In [8]:
def parse_bathrooms(text):
    if pd.isna(text):
        return np.nan, np.nan
    text_l = str(text).lower()
    is_shared = 1 if 'shared' in text_l else 0
    m = re.search(r'(\d+(\.\d+)?)', text_l)  # mencari angka dalam teks
    if m:
        num = float(m.group(1))
    elif 'half' in text_l:  # kata "half" = 0.5
        num = 0.5
    else:
        num = np.nan  # nilai kosong = NaN
    return num, is_shared

def parse_amenities(text):
    # hitung jumlah amenities dari string list Python.
    try:
        lst = ast.literal_eval(text)
        return len(lst)
    except Exception:
        return 0

def parse_verifications(text):
    """Hitung jumlah metode verifikasi host dari string list Python."""
    try:
        lst = ast.literal_eval(text)
        return len(lst)
    except Exception:
        return 0

def parse_pct(x):
    # mengubah string persentase menjadi float.
    if pd.isna(x):
        return np.nan
    return float(str(x).replace('%', ''))

def engineer(df):
    df = df.copy()

    # bathrooms
    baths = df['bathrooms_text'].apply(parse_bathrooms)
    df['n_bathrooms'] = baths.apply(lambda t: t[0])
    df['bath_shared'] = baths.apply(lambda t: t[1])

    # amenities count
    df['n_amenities'] = df['amenities'].apply(parse_amenities)

    # host verifications count
    df['n_host_verifications'] = df['host_verifications'].apply(parse_verifications)

    # percentages -> float
    df['host_response_rate']   = df['host_response_rate'].apply(parse_pct)
    df['host_acceptance_rate'] = df['host_acceptance_rate'].apply(parse_pct)

    # text length signals
    df['name_len'] = df['name'].fillna('').str.len()
    df['description_len'] = df['description'].fillna('').str.len()
    df['neighborhood_overview_len'] = df['neighborhood_overview'].fillna('').str.len()
    df['host_about_len'] = df['host_about'].fillna('').str.len()
    df['has_neighborhood_overview'] = df['neighborhood_overview'].notna().astype(int)
    df['has_host_about'] = df['host_about'].notna().astype(int)

    # boolean-ish string columns -> 0/1
    for col in ['instant_bookable', 'host_identity_verified']:
        df[col] = df[col].map({'t': 1, 'f': 0}).fillna(0).astype(int)

    # missing-review-score flag (no reviews yet)
    df['has_reviews'] = (df['number_of_reviews'] > 0).astype(int)

    # fill comment-derived columns for listings with zero comments
    df['comment_count'] = df['comment_count'].fillna(0)
    df['avg_comment_len'] = df['avg_comment_len'].fillna(0)
    df['n_unique_reviewers'] = df['n_unique_reviewers'].fillna(0)

    # ratio features
    df['beds_per_accommodates'] = df['beds'] / df['accommodates'].replace(0, np.nan)
    df['bath_per_accommodates'] = df['n_bathrooms'] / df['accommodates'].replace(0, np.nan)

    return df

train = engineer(train)
test  = engineer(test)

train.isnull().sum()


id                              0
price                           0
name                            0
description                    55
neighborhood_overview        1590
                             ... 
has_neighborhood_overview       0
has_host_about                  0
has_reviews                     0
beds_per_accommodates         690
bath_per_accommodates           6
Length: 62, dtype: int64

## 5. Menentukan Kolom Fitur (Kategorikal & Numerik)

Kolom kategorikal diubah ke dtype `category`, dan kategori pada data `test`
disamakan dengan kategori yang ada di `train` agar konsisten saat inference.


In [9]:
categorical_cols = [
    'property_type', 'room_type', 'host_response_time',
    'neighbourhood_cleansed', 'neighbourhood_group_cleansed',
    'host_neighbourhood',
]

numeric_cols = [
    'accommodates', 'n_bathrooms', 'bath_shared', 'bedrooms', 'beds',
    'n_amenities', 'minimum_nights', 'maximum_nights',
    'instant_bookable', 'longitude', 'latitude',
    'minimum_minimum_nights', 'maximum_minimum_nights',
    'minimum_maximum_nights', 'maximum_maximum_nights',
    'availability_30', 'availability_60', 'availability_90', 'availability_365',
    'host_response_rate', 'host_acceptance_rate', 'host_total_listings_count',
    'n_host_verifications', 'host_identity_verified',
    'number_of_reviews', 'review_scores_rating', 'review_scores_accuracy',
    'review_scores_cleanliness', 'review_scores_checkin',
    'review_scores_communication', 'review_scores_location',
    'review_scores_value', 'reviews_per_month', 'has_reviews',
    'name_len', 'description_len', 'neighborhood_overview_len',
    'host_about_len', 'has_neighborhood_overview', 'has_host_about',
    'comment_count', 'avg_comment_len', 'n_unique_reviewers',
    'beds_per_accommodates', 'bath_per_accommodates',
]

feature_cols = categorical_cols + numeric_cols

for c in categorical_cols:
    train[c] = train[c].astype('category')
    # samakan kategori test dengan train
    test[c] = pd.Categorical(test[c], categories=train[c].cat.categories)

X = train[feature_cols]
y_raw = train['price'].values


## 6. Membersihkan Target (`price`) & Log-Transform

Baris dengan harga `<= 0` dibuang karena tidak valid. Target di-transform
dengan `log1p` agar distribusinya lebih mendekati normal (umum untuk data
harga yang skewed).


In [10]:
mask_valid = train['price'] > 0
print(f'Dropping {(~mask_valid).sum()} rows with price <= 0 out of {len(train)}')
X = X[mask_valid.values]
y_raw = y_raw[mask_valid.values]
y = np.log1p(y_raw)

X_test = test[feature_cols]


Dropping 10 rows with price <= 0 out of 2954


## 7. Cross-Validation (KFold)

Evaluasi performa model dengan **5-fold KFold cross-validation**, menggunakan
metrik **MAPE (Mean Absolute Percentage Error)** — metrik yang sama dengan
yang dipakai Kaggle untuk menilai leaderboard. Prediksi dikembalikan ke skala
harga asli (`expm1`) sebelum dihitung MAPE-nya.


In [11]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
mapes = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(X)):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    model = LGBMRegressor(
        max_depth=6,
        num_leaves=31,           # kontrol kompleksitas utama LightGBM (~2^max_depth)
        learning_rate=0.05,
        n_estimators=400,
        reg_lambda=1.0,          # regularisasi L2
        random_state=42,
        verbose=-1,               # matikan log training LightGBM
    )
    model.fit(
        X_tr, y_tr,
        categorical_feature=categorical_cols,
    )
    pred_log = model.predict(X_va)
    pred = np.expm1(pred_log)
    true = np.expm1(y_va)

    mape = mean_absolute_percentage_error(true, pred)
    mapes.append(mape)
    print(f'Fold {fold+1}: MAPE={mape:.4f} ({mape*100:.2f}%)')

print(f'\nMean CV MAPE: {np.mean(mapes):.4f} (+/- {np.std(mapes):.4f})')
print(f'Mean CV MAPE: {np.mean(mapes)*100:.2f}% (+/- {np.std(mapes)*100:.2f}%)')


Fold 1: MAPE=0.2841 (28.41%)
Fold 2: MAPE=0.4055 (40.55%)
Fold 3: MAPE=0.3502 (35.02%)
Fold 4: MAPE=0.5677 (56.77%)
Fold 5: MAPE=0.3499 (34.99%)

Mean CV MAPE: 0.3915 (+/- 0.0961)
Mean CV MAPE: 39.15% (+/- 9.61%)


## 8. Training Model Final & Membuat Submission


In [12]:
import os
os.makedirs('../results', exist_ok=True)

final_model = LGBMRegressor(
    max_depth=6,
    num_leaves=31,
    learning_rate=0.05,
    n_estimators=400,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1,
)
final_model.fit(
    X, y,
    categorical_feature=categorical_cols,
)

test_pred_log = final_model.predict(X_test)
test_pred = np.expm1(test_pred_log)
test_pred = np.clip(test_pred, 0, None)  # harga tidak boleh negatif

submission = pd.DataFrame({
    'id': test_raw['id'],
    'price': test_pred
})
submission.to_csv('/kaggle/working/submission.csv', index=False)
print('\nSaved submission to /kaggle/working/submission.csv')
print(submission.head())



Saved submission to /kaggle/working/submission.csv
                    id        price
0             39326791    63.447643
1  1454846206172103024   300.503143
2   711238794811729789   343.091656
3             17519803    75.042116
4  1334579376251132916  1362.340868
